# 08 — The Story

**Thesis.** Amazon's *Verified Purchase* flag is a convenient but **imperfect** proxy for review trustworthiness. We can build a classifier that predicts it from language + behavior, but when we hold those signals up to a **real labeled fake-review dataset**, only some survive — and a few would actively *mislead* a platform that trusted the proxy.

*(Headline numbers below are from the executed analysis on the Amazon Reviews 2023 `Subscription_Boxes` category, 16,216 reviews, plus the Hollenbeck et al. ground-truth set. Re-run notebooks 01–07 to refresh.)*

## The question
**Primary:** what linguistic & behavioral signals distinguish verified-purchase from unverified reviews, and can they flag suspicious reviews the verification flag misses?

**Validation:** when checked against real fake-review labels, which signals actually survive — and where does the proxy mislead?

## Data & the weak proxy
- **Source:** McAuley/UCSD Amazon Reviews 2023 (ToS-safe), unified 16-column schema, product-metadata join; the same schema the polite scraper would emit.
- **Proxy label:** Verified Purchase — *purchase verification, not deception.* 88% verified here.
- **Strengthened** with behavioral signals: review bursts, rating skew, near-duplicate text, helpful-vote ratios, reviewer history breadth.
- **Ground truth:** Hollenbeck et al. (MIT), 381,734 labeled reviews, mapped onto the *same* schema.

## What we found

| Result | Value |
|---|---|
| Unverified reviews are **longer** (median words, verified→unverified) | 28 → **53** |
| Trust classifier on the proxy (random forest, ROC-AUC) | **0.80** |
| Reality check on **real** fakes (rating-independent `labeled`, RF ROC-AUC) | **0.72** |
| Same, `primary` label (defines fake via 5-star → circular) | 0.99 ⚠️ |

**Signal survival (proxy vs real label):**
- ✅ **Survives both** — `char_count`, `word_count`: review **length/effort** separates unverified reviews *and* real fakes.
- ⚠️ **Proxy-only (misleading)** — `helpful_votes`, `helpful_votes_pz`, `sentiment_neg`: track the verified flag but **not** real deception.
- The rest are weak under the honest label.

## Figures (poster set)
Exported to `reports/figures/` and embedded in the chapter notebooks.

### Proxy-label EDA (notebook 02)
![class balance](../reports/figures/02_class_balance.png)
![rating by proxy](../reports/figures/02_rating_by_proxy.png)
![length by proxy](../reports/figures/02_length_by_proxy.png)

### Trust classifier (notebook 04)
![ROC](../reports/figures/04_roc.png)
![feature importances](../reports/figures/04_importances.png)

### Styles & network (notebook 05)
![PCA](../reports/figures/05_pca.png)
![style terms](../reports/figures/05_style_terms.png)

### Ground-truth validation (notebook 07)
![real ROC](../reports/figures/07_real_roc.png)
![signal survival](../reports/figures/07_survival.png)

## Real-world impact & honest limitations
**Impact.** A platform could surface *low-trust* reviews the verified flag alone misses — but should weight **review-effort/length** signals over helpful-vote or negative-sentiment cues, which our ground-truth check shows are proxy artifacts.

**Limitations (stated plainly):**
- Verified Purchase ≠ deception; it's a *weak proxy*, and ground truth shows where it breaks.
- The `primary` label's 5-star circularity inflates apparent performance — we report the rating-independent `labeled` result as the honest one.
- One category here; broaden across categories (the notebooks auto-detect added data).
- In-scope methods only (no transformers / formal hypothesis tests).

**For the 3-min talk:** question → weak proxy → classifier (~0.80) → reality check (~0.72) → *length survives, helpful-votes mislead* → the verified flag is useful but imperfect.